<a href="https://colab.research.google.com/github/lakshitparmar/HCL-Project-by-Lakshit/blob/main/Hospital_Appointment_Operations_Analysis_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 9: Hospital Appointment Operations Analysis
**Industry:** Healthcare Operations  |  **Tools:** Python, Pandas, NumPy, Matplotlib

**Business scenario:** Analyze appointment attendance, waiting times and department workload to identify operational bottlenecks.
> This is an **operational** dataset, not a clinical dataset. No medical conclusions are drawn.

**How to use in Google Colab:** Run cells top to bottom. When asked, upload `09_hospital_appointments.csv`.
If no file is uploaded, a small *synthetic demo dataset* is generated so the notebook still runs end to end.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)
plt.rcParams['figure.figsize'] = (9, 4.5)

## 1. Load the data (Task 124)

In [ ]:
import os
FILE = '09_hospital_appointments.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        print('Please upload', FILE, '(or cancel to use demo data)')
        up = files.upload()
        if up:
            FILE = list(up.keys())[0]
    except Exception as e:
        print('Upload skipped:', e)

if os.path.exists(FILE):
    df = pd.read_csv(FILE)
    print('Loaded real file:', FILE)
else:
    # ---- synthetic DEMO data (only used when the real CSV is missing) ----
    rng = np.random.default_rng(42)
    n = 1500
    depts = ['Cardiology','Orthopedics','Pediatrics','Dermatology','General Medicine','ENT','Gynecology']
    fee_map = dict(zip(depts,[900,800,500,600,400,550,700]))
    d = rng.choice(depts, n, p=[.12,.14,.15,.12,.27,.10,.10])
    sms = rng.choice(['Yes','No'], n, p=[.6,.4])
    p_noshow = np.where(sms=='Yes', .12, .28)
    status = np.where(rng.random(n) < p_noshow, 'No-Show', 'Attended')
    wait = np.clip(rng.gamma(4, 8, n), 2, None).round()
    wait[rng.choice(n, 12, replace=False)] += rng.integers(120, 240, 12)   # extreme waits
    df = pd.DataFrame({
        'Appointment_ID': [f'APT{i:05d}' for i in range(1, n+1)],
        'Patient_ID': rng.integers(1000, 1600, n),
        'Department': d,
        'Appointment_Date': pd.Timestamp('2026-01-01') + pd.to_timedelta(rng.integers(0, 90, n), unit='D'),
        'Appointment_Type': rng.choice(['New','Follow-up'], n, p=[.4,.6]),
        'Waiting_Time': wait,
        'SMS_Reminder': sms,
        'Attendance_Status': status,
        'Consultation_Fee': [fee_map[x] for x in d],
    })
    df.loc[rng.choice(n, 60, replace=False), 'Waiting_Time'] = np.nan      # missing waits
    df = pd.concat([df, df.sample(8, random_state=1)], ignore_index=True)   # duplicate IDs
    print('Using SYNTHETIC demo data')

Please upload 09_hospital_appointments.csv (or cancel to use demo data)


### Column mapping
Edit the right-hand names below if your CSV uses different column headings.

In [ ]:
COL = {
    'id'     : 'Appointment_ID',
    'dept'   : 'Department',
    'date'   : 'Appointment_Date',
    'type'   : 'Appointment_Type',     # New / Follow-up
    'wait'   : 'Waiting_Time',         # minutes
    'sms'    : 'SMS_Reminder',         # Yes / No
    'status' : 'Attendance_Status',    # Attended / No-Show
    'fee'    : 'Consultation_Fee',
}
missing_cols = [v for v in COL.values() if v not in df.columns]
print('Columns not found in data:', missing_cols if missing_cols else 'None - mapping OK')
print('Available columns:', list(df.columns))

In [ ]:
print('Shape:', df.shape)
display(df.head())
df.info()
display(df.describe(include='all').T)

## 2. Missing waiting times (Task 125)

In [ ]:
df[COL['wait']] = pd.to_numeric(df[COL['wait']], errors='coerce')
print('Missing values per column:')
print(df.isna().sum())

n_missing = df[COL['wait']].isna().sum()
print(f"\nMissing waiting times: {n_missing} ({n_missing/len(df)*100:.2f}%)")

# Handling: keep a flag, then impute with the department MEDIAN (robust to extreme waits)
df['Wait_Was_Missing'] = df[COL['wait']].isna()
df['Wait_Clean'] = df[COL['wait']].fillna(df.groupby(COL['dept'])[COL['wait']].transform('median'))
df['Wait_Clean'] = df['Wait_Clean'].fillna(df[COL['wait']].median())   # fallback
print('Missing after handling:', df['Wait_Clean'].isna().sum())

## 3. Duplicate Appointment_ID values (Task 126)

In [ ]:
dup_mask = df.duplicated(subset=COL['id'], keep=False)
print('Rows involved in duplicate IDs:', dup_mask.sum())
print('Fully identical duplicate rows :', df.duplicated().sum())
display(df[dup_mask].sort_values(COL['id']).head(10))

before = len(df)
df = df.drop_duplicates(subset=COL['id'], keep='first').reset_index(drop=True)
print(f'Removed {before-len(df)} duplicate rows. Clean shape: {df.shape}')

## 4. Total appointments and attendance-status percentages (Task 127)

In [ ]:
df[COL['status']] = df[COL['status']].astype(str).str.strip()
total = len(df)
status_counts = df[COL['status']].value_counts()
status_pct = (status_counts / total * 100).round(2)
attendance_summary = pd.DataFrame({'Count': status_counts, 'Percentage': status_pct})
print('Total appointments:', total)
display(attendance_summary)
status_counts.plot(kind='bar', title='Appointments by attendance status', color=['#2a9d8f','#e76f51','#e9c46a'][:len(status_counts)])
plt.ylabel('Appointments'); plt.xticks(rotation=0); plt.show()

# helper flags used in later tasks
s = df[COL['status']].str.lower()
df['Is_NoShow']   = s.str.contains('no') & ~s.str.contains('attend')
df['Is_Attended'] = s.str.contains('attend') & ~s.str.contains('no')

## 5. Average waiting time by department (Task 128)

In [ ]:
avg_wait = (df.groupby(COL['dept'])['Wait_Clean'].agg(Avg_Wait='mean', Appointments='count')
              .round(2).sort_values('Avg_Wait', ascending=False))
display(avg_wait)
avg_wait['Avg_Wait'].plot(kind='barh', title='Average waiting time by department (min)', color='#264653')
plt.gca().invert_yaxis(); plt.xlabel('Minutes'); plt.show()

## 6. Departments with the highest no-show percentage (Task 129)

In [ ]:
dept_noshow = (df.groupby(COL['dept'])
                 .agg(Appointments=(COL['id'],'count'), NoShows=('Is_NoShow','sum')))
dept_noshow['NoShow_%'] = (dept_noshow['NoShows']/dept_noshow['Appointments']*100).round(2)
dept_noshow = dept_noshow.sort_values('NoShow_%', ascending=False)
display(dept_noshow)
print('Highest no-show department:', dept_noshow.index[0], f"({dept_noshow['NoShow_%'].iloc[0]}%)")
dept_noshow['NoShow_%'].plot(kind='bar', title='No-show % by department', color='#e76f51')
plt.ylabel('%'); plt.xticks(rotation=30, ha='right'); plt.show()

## 7. No-show rate: SMS reminder Yes vs No (Task 130)

In [ ]:
ct_counts = pd.crosstab(df[COL['sms']], df[COL['status']], margins=True)
ct_pct    = (pd.crosstab(df[COL['sms']], df[COL['status']], normalize='index')*100).round(2)
print('Counts'); display(ct_counts)
print('Row percentages (%)'); display(ct_pct)

sms_noshow = df.groupby(COL['sms'])['Is_NoShow'].mean().mul(100).round(2)
print('\nNo-show rate by SMS reminder (%):'); print(sms_noshow)
sms_noshow.plot(kind='bar', title='No-show rate by SMS reminder', color=['#e9c46a','#2a9d8f'])
plt.ylabel('%'); plt.xticks(rotation=0); plt.show()

## 8. Average consultation fee by department (Task 131)

In [ ]:
df[COL['fee']] = pd.to_numeric(df[COL['fee']], errors='coerce')
avg_fee = df.groupby(COL['dept'])[COL['fee']].agg(Avg_Fee='mean', Appointments='count').round(2).sort_values('Avg_Fee', ascending=False)
display(avg_fee)

## 9. Appointment volume by date and department (Task 132)

In [ ]:
df[COL['date']] = pd.to_datetime(df[COL['date']], errors='coerce')
vol_pivot = pd.crosstab(df[COL['date']].dt.date, df[COL['dept']])
print('Pivot (first 10 days):'); display(vol_pivot.head(10))

daily = df.groupby(df[COL['date']].dt.date).size()
print('Busiest date:', daily.idxmax(), '->', daily.max(), 'appointments')
daily.plot(title='Daily appointment volume'); plt.ylabel('Appointments'); plt.xlabel('Date'); plt.show()

weekly = df.groupby([pd.Grouper(key=COL['date'], freq='W'), COL['dept']]).size().unstack(fill_value=0)
weekly.plot(title='Weekly volume by department'); plt.legend(bbox_to_anchor=(1.02,1), loc='upper left'); plt.ylabel('Appointments'); plt.show()

## 10. Waiting times: new vs follow-up (Task 133)

In [ ]:
type_wait = df.groupby(COL['type'])['Wait_Clean'].agg(['count','mean','median','std']).round(2)
display(type_wait)
df.boxplot(column='Wait_Clean', by=COL['type'], grid=False)
plt.title('Waiting time by appointment type'); plt.suptitle(''); plt.ylabel('Minutes'); plt.show()

## 11. NumPy statistics: median, quartiles, standard deviation (Task 134)

In [ ]:
w = df['Wait_Clean'].to_numpy()
median = np.median(w)
q1, q3 = np.percentile(w, [25, 75])
std = np.std(w, ddof=1)
print(f'Mean   : {np.mean(w):.2f} min')
print(f'Median : {median:.2f} min')
print(f'Q1     : {q1:.2f} min')
print(f'Q3     : {q3:.2f} min')
print(f'Std Dev: {std:.2f} min')

## 12. Unusually long waiting times using IQR (Task 135)

In [ ]:
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
print(f'IQR = {iqr:.2f} | Lower fence = {lower:.2f} | Upper fence = {upper:.2f}')

outliers = df[df['Wait_Clean'] > upper]
print(f'Unusually long waits: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)')
display(outliers.sort_values('Wait_Clean', ascending=False).head(10))
print('Outliers by department:'); display(outliers[COL['dept']].value_counts())

plt.boxplot(w, vert=False); plt.axvline(upper, color='red', ls='--', label='Upper fence')
plt.title('Waiting-time distribution'); plt.xlabel('Minutes'); plt.legend(); plt.show()

## 13. Waiting-time categories (Task 136)

In [ ]:
bins   = [-np.inf, 15, 30, 60, np.inf]
labels = ['Short (0-15)', 'Moderate (16-30)', 'Long (31-60)', 'Very Long (60+)']
df['Wait_Category'] = pd.cut(df['Wait_Clean'], bins=bins, labels=labels)

wait_bands = df['Wait_Category'].value_counts().reindex(labels)
wait_bands = pd.DataFrame({'Count': wait_bands, 'Percentage': (wait_bands/len(df)*100).round(2)})
display(wait_bands)
display(pd.crosstab(df[COL['dept']], df['Wait_Category']))
wait_bands['Count'].plot(kind='bar', title='Waiting-time bands', color='#457b9d'); plt.xticks(rotation=15); plt.show()

## 14. Estimated total consultation fees for attended appointments (Task 137)

In [ ]:
attended = df[df['Is_Attended']]
fee_summary = (attended.groupby(COL['dept'])[COL['fee']]
                       .agg(Attended_Appointments='count', Estimated_Total_Fees='sum')
                       .sort_values('Estimated_Total_Fees', ascending=False))
fee_summary.loc['TOTAL'] = fee_summary.sum()
display(fee_summary)

lost = df[df['Is_NoShow']][COL['fee']].sum()
print(f'Estimated fee value of no-show appointments (potential lost revenue): {lost:,.0f}')

## 15. Five operational recommendations (Task 138)

In [ ]:
top_ns   = dept_noshow.index[0]
top_wait = avg_wait.index[0]
busiest  = vol_pivot.sum().idxmax()
yes_rate = sms_noshow.get('Yes', np.nan); no_rate = sms_noshow.get('No', np.nan)
long_pct = (df['Wait_Clean'] > 60).mean()*100

recs = [
 f"1. Targeted no-show reduction: {top_ns} has the highest no-show rate ({dept_noshow['NoShow_%'].iloc[0]}%). Add double reminders and easy rescheduling there first.",
 f"2. Expand SMS reminders: no-show rate is {yes_rate}% with SMS vs {no_rate}% without. Make reminders standard for all departments.",
 f"3. Cut waiting bottlenecks: {top_wait} has the longest average wait ({avg_wait['Avg_Wait'].iloc[0]} min). Review staffing, slot length and check-in flow.",
 f"4. Balance workload: {busiest} carries the highest appointment volume. Shift staff or open extra slots on peak dates (busiest date: {daily.idxmax()}).",
 f"5. Monitor extreme waits: {len(outliers)} appointments exceed the IQR fence ({upper:.0f} min) and {long_pct:.1f}% wait over 60 min. Set an alert threshold and review these cases weekly.",
]
for r in recs: print(r, end='\n\n')

## 16. Export outputs

In [ ]:
os.makedirs('outputs', exist_ok=True)
avg_wait.to_csv('outputs/avg_wait_by_department.csv')
dept_noshow.to_csv('outputs/noshow_by_department.csv')
ct_pct.to_csv('outputs/sms_vs_attendance.csv')
avg_fee.to_csv('outputs/avg_fee_by_department.csv')
wait_bands.to_csv('outputs/waiting_time_bands.csv')
fee_summary.to_csv('outputs/estimated_fees_attended.csv')
df.to_csv('outputs/cleaned_appointments.csv', index=False)
open('outputs/recommendations.txt','w').write('\n\n'.join(recs))
print('Saved to /outputs:', os.listdir('outputs'))

try:
    import shutil; shutil.make_archive('project9_outputs','zip','outputs')
    from google.colab import files; files.download('project9_outputs.zip')
except Exception:
    pass

## Notes for SRS / Submission
- **Assumptions:** Waiting time is in minutes; missing waits imputed with department median (flagged in `Wait_Was_Missing`); duplicate IDs keep the first record.
- **Limitations:** Operational data only; no clinical or medical conclusions. Fee totals are estimates based on `Consultation_Fee`.
- **Submit:** SRS document + this `.ipynb` / `.py` file + GitHub repo link on the LMS.